In [0]:
%sql
create or replace table dlt_catalog.raw.toll_log(
  txn_id INT,
  vehicle_no VARCHAR(55),
  vehicle_type VARCHAR(55),
  crossing_time TIMESTAMP
);
INSERT INTO dlt_catalog.raw.toll_log (txn_id, vehicle_no, vehicle_type, crossing_time)
VALUES
(1, 'DL01AA1111', 'car',        '2026-01-10 08:00:00'),
(2, 'DL01AA1111', 'car',        '2026-01-10 11:00:00'),
(3, 'DL02BB2222', 'truck',      '2026-01-10 09:00:00'),
(4, 'DL02BB2222', 'truck',      '2026-01-10 16:00:00'),
(5, 'DL03CC3333', 'bus',        '2026-01-10 10:00:00'),
(6, 'DL03CC3333', 'bus',        '2026-01-10 12:00:00'),
(7, 'DL04DD4444', 'motorcycle', '2026-01-10 11:00:00'),
(8, 'DL05EE5555', 'car',        '2026-01-10 14:00:00');



You are given a table toll_log which stores every vehicle crossing a toll plaza.
Each record represents one crossing of a vehicle through the toll gate.

###Write a SQL query to find the total toll collected per day.


###Toll Rule:

    Motorcycle → Free (₹0)

    Car → ₹40

    Bus → ₹70

    Truck → ₹80

    Return Rule (within 4 hours)

Note : If the same vehicle returns within 4 hours, charge a reduced toll:

    Car → ₹20

    Bus → ₹30

    Truck → ₹40

If the vehicle returns after 4 hours, charge the full toll again.

In [0]:
%sql
select * from dlt_catalog.raw.toll_log

In [0]:
%sql
select sum(amount) from (
select crossing_time,
case 
-- Motorcycle free
when vehicle_type = 'motorcycle' then 0
-- First crossing (no previous record)
when time_diff is null then
        CASE
          WHEN vehicle_type = 'car' THEN 40
          WHEN vehicle_type = 'bus' THEN 70
          WHEN vehicle_type = 'truck' THEN 80
        END
-- Return within 4 hours (discounted toll)
when vehicle_type = 'car' and time_diff < 4 then 20 
when vehicle_type = 'bus' and time_diff < 4 then 30
when vehicle_type = 'truck' and time_diff < 4 then 40
-- Return after 4 hours (full toll)  
when vehicle_type = 'car' and time_diff > 4 then 40 
when vehicle_type = 'bus' and time_diff > 4 then 70
when vehicle_type = 'truck' and time_diff > 4 then 80
end as Amount from (
SELECT
  txn_id,
  vehicle_no,
  vehicle_type,
  crossing_time,
  TIMESTAMPDIFF(
    hour,
    LAG(crossing_time) OVER (
      PARTITION BY vehicle_no
      ORDER BY crossing_time
    ),
    crossing_time
  ) AS time_diff
FROM dlt_catalog.raw.toll_log
))
